# Sequence Models — MLP, LSTM, GRU, Attention, Transformer

Train track-level emotion classifiers on per-track MFCC sequences.

**Design notes:**
- One MFCC sequence per track (`n_frames` × `n_mfcc`).
- Train/val/test splits are **track-level** (same `song_id` split as static baselines).
- No random frame-level splitting.
- Metrics saved to `results/metrics/sequence_models_summary.csv`.

In [1]:
from pathlib import Path
import logging
import sys

import pandas as pd
import torch

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_configs, resolve_path
from src.features.mfcc_sequences import extract_mfcc_sequences_dataset, load_mfcc_manifest
from src.training.train_sequence import log_training_device, resolve_device, train_and_evaluate_sequence_models

configs = load_configs(PROJECT_ROOT)
root = get_project_root(PROJECT_ROOT)

if torch.cuda.is_available():
    configs["training"]["general"]["device"] = "cuda"
    print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}")
else:
    print(
        "WARNING: CUDA not available — sequence models will train on CPU. "
        "Install a CUDA-enabled PyTorch build for your RTX 4060 Ti."
    )

device = resolve_device(str(configs["training"]["general"]["device"]))
log_training_device(device)

22:02:51 | INFO | src.training.train_sequence | Training device: cuda (NVIDIA GeForce RTX 4060 Ti)


PyTorch 2.11.0+cu128 | CUDA 12.8 | NVIDIA GeForce RTX 4060 Ti


## 1. Extract MFCC sequences (skipped if manifest exists)

In [2]:
manifest_path = resolve_path(root, configs["paths"]["features"]["sequence_mfcc_manifest"])

if manifest_path.exists():
    manifest = load_mfcc_manifest(configs)
    print(f"Loaded existing MFCC manifest: {manifest_path} ({len(manifest)} tracks)")
else:
    from src.data.load_deam import build_metadata_table
    metadata = build_metadata_table(configs)
    manifest = extract_mfcc_sequences_dataset(metadata, configs)
    print(f"Extracted MFCC sequences for {len(manifest)} tracks")

manifest.head()

22:03:01 | INFO | src.data.load_deam | Found dynamic annotation folder: C:\Users\athen\Desktop\music-emotion-recognition\data\raw\DEAM\DEAM_annotations\annotations\annotations_averaged_per_song\dynamic (per second annotations)
22:03:01 | INFO | src.data.load_deam | Found dynamic annotation folder: C:\Users\athen\Desktop\music-emotion-recognition\data\raw\DEAM\DEAM_annotations\annotations\annotations_averaged_per_song\dynamic (per second annotations)
22:03:01 | INFO | src.data.load_deam | Loaded static annotations from static_annotations_averaged_songs_1_2000.csv (1744 rows)
22:03:01 | INFO | src.data.load_deam | Loaded static annotations from static_annotations_averaged_songs_2000_2058.csv (58 rows)
22:03:01 | INFO | src.data.load_deam | Found dynamic annotation folder: C:\Users\athen\Desktop\music-emotion-recognition\data\raw\DEAM\DEAM_annotations\annotations\annotations_averaged_per_song\dynamic (per second annotations)
22:03:01 | INFO | src.data.load_deam | Dynamic annotation folder

Scanning audio files:   0%|          | 0/1802 [00:00<?, ?it/s]

Extracting MFCC sequences:   0%|          | 0/1802 [00:00<?, ?it/s]

22:08:17 | INFO | src.features.mfcc_sequences | Saved MFCC manifest to C:\Users\athen\Desktop\music-emotion-recognition\data\features\sequence\mfcc_manifest.csv (1802 tracks)


Extracted MFCC sequences for 1802 tracks


,song_id,n_frames,n_mfcc,sequence_path
0,2,1941,20,mfcc\2.npy
1,3,1940,20,mfcc\3.npy
2,4,1940,20,mfcc\4.npy
3,5,1940,20,mfcc\5.npy
4,7,1941,20,mfcc\7.npy


## 2. Train sequence models

In [3]:
summary_df = train_and_evaluate_sequence_models(
    configs,
    model_names=("mlp", "lstm", "gru", "attention", "transformer"),
    extract_features=False,
)
summary_df

22:10:51 | INFO | src.training.train_sequence | Training device: cuda (NVIDIA GeForce RTX 4060 Ti)
22:10:51 | INFO | src.data.load_deam | Found dynamic annotation folder: C:\Users\athen\Desktop\music-emotion-recognition\data\raw\DEAM\DEAM_annotations\annotations\annotations_averaged_per_song\dynamic (per second annotations)
22:10:51 | INFO | src.data.load_deam | Found dynamic annotation folder: C:\Users\athen\Desktop\music-emotion-recognition\data\raw\DEAM\DEAM_annotations\annotations\annotations_averaged_per_song\dynamic (per second annotations)
22:10:51 | INFO | src.data.load_deam | Loaded static annotations from static_annotations_averaged_songs_1_2000.csv (1744 rows)
22:10:51 | INFO | src.data.load_deam | Loaded static annotations from static_annotations_averaged_songs_2000_2058.csv (58 rows)
22:10:51 | INFO | src.data.load_deam | Found dynamic annotation folder: C:\Users\athen\Desktop\music-emotion-recognition\data\raw\DEAM\DEAM_annotations\annotations\annotations_averaged_per_son

Scanning audio files:   0%|          | 0/1802 [00:00<?, ?it/s]

22:11:00 | INFO | src.training.train_sequence | Sequence split validation passed (track-level, no frame leakage).


Training sequence models:   0%|          | 0/5 [00:00<?, ?it/s]

22:11:00 | INFO | src.training.train_sequence | Training sequence model: mlp on cuda
22:11:11 | INFO | src.training.train_sequence | mlp epoch 1/50 — train_loss=2.1993, val_macro_f1=0.3341
22:11:13 | INFO | src.training.train_sequence | mlp epoch 2/50 — train_loss=1.2117, val_macro_f1=0.3468
22:11:14 | INFO | src.training.train_sequence | mlp epoch 3/50 — train_loss=1.1404, val_macro_f1=0.3648
22:11:15 | INFO | src.training.train_sequence | mlp epoch 4/50 — train_loss=1.1167, val_macro_f1=0.3800
22:11:16 | INFO | src.training.train_sequence | mlp epoch 5/50 — train_loss=1.0971, val_macro_f1=0.3622
22:11:17 | INFO | src.training.train_sequence | mlp epoch 6/50 — train_loss=1.0899, val_macro_f1=0.3773
22:11:18 | INFO | src.training.train_sequence | mlp epoch 7/50 — train_loss=1.0785, val_macro_f1=0.3985
22:11:20 | INFO | src.training.train_sequence | mlp epoch 8/50 — train_loss=1.0541, val_macro_f1=0.3756
22:11:21 | INFO | src.training.train_sequence | mlp epoch 9/50 — train_loss=1.0551,

,model_name,task_type,feature_type,target_type,eval_split,train_size,val_size,test_size,accuracy,macro_f1,weighted_f1,best_val_macro_f1
0,mlp,sequence,mfcc_sequence,emotion_quadrant,val,1261,270,271,0.633333,0.398539,0.551217,0.398539
1,mlp,sequence,mfcc_sequence,emotion_quadrant,test,1261,270,271,0.605166,0.391040,0.531111,0.398539
2,lstm,sequence,mfcc_sequence,emotion_quadrant,val,1261,270,271,0.629630,0.530145,0.612141,0.530145
3,lstm,sequence,mfcc_sequence,emotion_quadrant,test,1261,270,271,0.586716,0.470715,0.569017,0.530145
4,gru,sequence,mfcc_sequence,emotion_quadrant,val,1261,270,271,0.614815,0.506241,0.600611,0.506241
5,gru,sequence,mfcc_sequence,emotion_quadrant,test,1261,270,271,0.527675,0.393232,0.503992,0.506241
6,attention,sequence,mfcc_sequence,emotion_quadrant,val,1261,270,271,0.629630,0.529246,0.611179,0.529246
7,attention,sequence,mfcc_sequence,emotion_quadrant,test,1261,270,271,0.564576,0.450535,0.545034,0.529246
8,transformer,sequence,mfcc_sequence,emotion_quadrant,val,1261,270,271,0.596296,0.480316,0.579514,0.480316
9,transformer,sequence,mfcc_sequence,emotion_quadrant,test,1261,270,271,0.594096,0.509829,0.586491,0.480316


## 3. Test-set results

In [4]:
test_summary = summary_df[summary_df["eval_split"] == "test"].sort_values("macro_f1", ascending=False)
test_summary[["model_name", "accuracy", "macro_f1", "weighted_f1", "best_val_macro_f1"]]

,model_name,accuracy,macro_f1,weighted_f1,best_val_macro_f1
9,transformer,0.594096,0.509829,0.586491,0.480316
3,lstm,0.586716,0.470715,0.569017,0.530145
7,attention,0.564576,0.450535,0.545034,0.529246
5,gru,0.527675,0.393232,0.503992,0.506241
1,mlp,0.605166,0.391040,0.531111,0.398539


## 4. Saved outputs

In [5]:
metrics_dir = resolve_path(root, configs["paths"]["results"]["metrics"])
figures_dir = resolve_path(root, configs["paths"]["reports"]["figures"])
print("Summary:", metrics_dir / "sequence_models_summary.csv")
print("Per-model metrics: sequence_{model}_{split}_metrics.json")
print("Confusion matrices: reports/figures/sequence_{model}_confusion_matrix.png")

Summary: C:\Users\athen\Desktop\music-emotion-recognition\results\metrics\sequence_models_summary.csv
Per-model metrics: sequence_{model}_{split}_metrics.json
Confusion matrices: reports/figures/sequence_{model}_confusion_matrix.png
